In [1]:
# Run this cell to install libraries
!pip install fastapi uvicorn python-dotenv redis -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:


import os
from contextlib import asynccontextmanager
from fastapi import FastAPI, Depends
import redis.asyncio as aioredis
from dotenv import load_dotenv
import nest_asyncio
import uvicorn

nest_asyncio.apply()
load_dotenv()

# ==================== Redis Cloud Config ====================
REDIS_HOST = os.getenv("REDIS_HOST")
REDIS_PORT = int(os.getenv("REDIS_PORT"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD")
REDIS_USERNAME = os.getenv("REDIS_USERNAME", "default")

print(f"🔗 Connecting to Redis Cloud: {REDIS_HOST}:{REDIS_PORT}")

# ==================== Redis Connection Pool (Without SSL) ====================
redis_pool = None

@asynccontextmanager
async def lifespan(app: FastAPI):
    global redis_pool
    
    # ✅ Correct way for your Redis Cloud instance (No SSL)
    redis_pool = aioredis.ConnectionPool(
        host=REDIS_HOST,
        port=REDIS_PORT,
        username=REDIS_USERNAME,
        password=REDIS_PASSWORD,
        decode_responses=True,
        max_connections=30
        # ssl=True nahi lagaya kyunki aapka instance non-TLS hai
    )
    print("✅ Redis Cloud Connected Successfully!")
    yield
    await redis_pool.aclose()

app = FastAPI(title="FastAPI + Redis Cloud", lifespan=lifespan)

async def get_redis() -> aioredis.Redis:
    return aioredis.Redis(connection_pool=redis_pool)

# ==================== Endpoints ====================
@app.get("/")
async def root():
    return {"message": "FastAPI + Redis Cloud is working!"}

@app.get("/redis-test")
async def redis_test(r: aioredis.Redis = Depends(get_redis)):
    await r.set("foo", "Hello from Redis Cloud via FastAPI!", ex=300)
    value = await r.get("foo")
    return {"message": value}

@app.get("/cache/{key}")
async def get_cache(key: str, r: aioredis.Redis = Depends(get_redis)):
    cached = await r.get(key)
    if cached:
        return {"source": "redis_cloud", "data": cached}
    
    new_data = f"Fresh data for {key}"
    await r.set(key, new_data, ex=600)
    return {"source": "fresh", "data": new_data}

# ==================== Run Server ====================
print("🚀 Starting FastAPI server...")
uvicorn.run(app, host="0.0.0.0", port=8000)

INFO:     Started server process [20304]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 Connecting to Redis Cloud: redis-11344.c275.us-east-1-4.ec2.cloud.redislabs.com:11344
🚀 Starting FastAPI server...
✅ Redis Cloud Connected Successfully!
INFO:     127.0.0.1:4897 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:4897 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:3271 - "GET /redis-test HTTP/1.1" 200 OK
INFO:     127.0.0.1:4224 - "GET /cache/hello HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [20304]


In [3]:
!pip install fastapi uvicorn python-dotenv upstash-redis nest_asyncio -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:


import os
from fastapi import FastAPI
from upstash_redis import Redis
from dotenv import load_dotenv
import nest_asyncio
import uvicorn

nest_asyncio.apply()
load_dotenv()

# ==================== Upstash Redis Config ====================
UPSTASH_URL = os.getenv("UPSTASH_REDIS_REST_URL")
UPSTASH_TOKEN = os.getenv("UPSTASH_REDIS_REST_TOKEN")

print(f"🔗 Connecting to Upstash Redis...")

# ==================== Upstash Redis Client ====================
redis = Redis(url=UPSTASH_URL, token=UPSTASH_TOKEN)

print("✅ Upstash Redis Connected Successfully!")

# ==================== FastAPI App ====================
app = FastAPI(title="FastAPI + Upstash Redis")

# ==================== Endpoints ====================
@app.get("/")
async def root():
    return {"message": "FastAPI + Upstash Redis is working!"}

@app.get("/redis-test")
async def redis_test():
    redis.set("foo", "Hello from Upstash Redis!", ex=300)
    value = redis.get("foo")
    return {"message": value}

@app.get("/cache/{key}")
async def get_cache(key: str):
    cached = redis.get(key)
    if cached:
        return {"source": "upstash_cache", "data": cached}
    
    new_data = f"Fresh data for {key}"
    redis.set(key, new_data, ex=600)
    return {"source": "fresh", "data": new_data}

@app.get("/counter")
async def counter():
    count = redis.incr("visits")
    return {"visits": count}

# ==================== Run Server ====================
print("🚀 Starting FastAPI server...")
print("📌 Open: http://127.0.0.1:8000/docs")

uvicorn.run(app, host="0.0.0.0", port=8000)

🔗 Connecting to Upstash Redis...
✅ Upstash Redis Connected Successfully!
🚀 Starting FastAPI server...
📌 Open: http://127.0.0.1:8000/docs


INFO:     Started server process [20304]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:2805 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:2805 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:11886 - "GET /redis-test HTTP/1.1" 200 OK
INFO:     127.0.0.1:14427 - "GET /cache/hello HTTP/1.1" 200 OK


In [1]:
# 1. Install libraries
!pip install pymongo python-dotenv -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

import os
from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
import datetime

load_dotenv()

MONGODB_URI = os.getenv("MONGODB_URI")

client = None

def connect_to_mongodb():
    global client
    try:
        client = MongoClient(MONGODB_URI)
        client.admin.command('ping')
        print("✅ Connected to MongoDB successfully!")
        return client
    except ConnectionFailure as e:
        print("❌ Connection failed:", e)
        return None

def get_urls_collection():
    """URL Shortener ke liye collection return karega"""
    if client is None:
        connect_to_mongodb()
    db = client["url-shortener-project-db"]     # ← Tumhara database
    return db["urls"]                           # ← Collection name

# ==================== Usage Example ====================

client = connect_to_mongodb()

if client:
    urls_collection = get_urls_collection()

    # Example: Ek short URL save karna
    new_url = {
        "original_url": "https://www.google.com",
        "short_code": "abc123",
        "clicks": 0,
        "created_at": datetime.datetime.now()
    }

    result = urls_collection.insert_one(new_url)
    print(f"✅ URL saved with ID: {result.inserted_id}")

    # Example: Saare URLs dekhna
    print("\n📌 All Short URLs:")
    for url in urls_collection.find():
        print(url)

✅ Connected to MongoDB successfully!
✅ URL saved with ID: 6a47081d4192f69942cb1841

📌 All Short URLs:
{'_id': ObjectId('6a47081d4192f69942cb1841'), 'original_url': 'https://www.google.com', 'short_code': 'abc123', 'clicks': 0, 'created_at': datetime.datetime(2026, 7, 3, 6, 23, 49, 316000)}
